---
<a id="prefix"></a>
## 05 — Prefix Sums

| # | File | Difficulty | Key idea |
|---|------|------------|----------|
| 0019 | [running_sum.ipynb](../0019.running_sum.ipynb) | 🟢 | `accumulate` / iterative build |
| 0069 | [pivot_index.ipynb](../0069.pivot_index.ipynb) | 🟢 | `right = total − left − nums[i]` |
| 0023 | [product_except_self.ipynb](../0023.product_except_self.ipynb) | 🟡 | Prefix × suffix in-place |
| 0035 | [subarray_sum_k.ipynb](../0035.subarray_sum_k.ipynb) | 🟡 | Prefix + HashMap — query before store |
| 0158 | [range_sum_query_immutable.ipynb](../0158.range_sum_query_immutable.ipynb) | 🟢 | Build once, query ranges in O(1) |
| 0159 | [max_size_subarray_sum_k.ipynb](../0159.max_size_subarray_sum_k.ipynb) | 🟡 | Prefix first-seen index for longest length |
| 0160 | [contiguous_array.ipynb](../0160.contiguous_array.ipynb) | 🟡 | Map 0→-1, 1→+1; equal prefix means balanced |
| 0161 | [subarray_sums_divisible_by_k.ipynb](../0161.subarray_sums_divisible_by_k.ipynb) | 🟡 | Prefix mod frequency counts |
| 0162 | [binary_subarrays_with_sum.ipynb](../0162.binary_subarrays_with_sum.ipynb) | 🟡 | Prefix + freq map on binary sums |

## Prefix Sums — Mental Model

### The Core Idea: Precompute Cumulative Work

A prefix sum transforms a sequence of values into a sequence of **running totals**, so any range query becomes O(1) instead of O(n).

```
nums:   [3,  1,  4,  1,  5]
prefix: [3,  4,  8,  9, 14]

sum(i..j) = prefix[j] - prefix[i-1]
```

---

### How to Recognize It

Ask yourself: *"Am I repeatedly summing a subarray?"*  
If yes — stop looping, start accumulating.

Trigger phrases:
- "subarray sum equals k"
- "range sum query"
- "product of everything except self"
- "balance point / pivot"

---

### The Three Patterns

**1. Range query** — build prefix, then answer in O(1)
```python
prefix[j] - prefix[i-1]   # sum of nums[i..j]
```

**2. Balance / pivot** — split array into left and right halves
```python
right = total - left - nums[i]   # O(n), no extra array
```

**3. Prefix + HashMap** — count subarrays with a target sum
```python
# Query BEFORE you store — counts subarrays ending here
count += seen[prefix - k]
seen[prefix] += 1
```

---

### The HashMap Pattern in Detail

The key insight: `sum(i..j) == k` iff `prefix[j] - prefix[i-1] == k`, i.e., `prefix[i-1] == prefix[j] - k`.

So at each index, ask *"how many past prefixes equal current − k?"* — then record the current prefix for future queries.

```
Query first → Store second
(otherwise you count the subarray against itself)
```

---

### Common Gotchas

| Pitfall | Fix |
|---|---|
| Off-by-one on range queries | Use 1-indexed prefix array, seed `prefix[0] = 0` |
| HashMap misses empty subarray | Seed `seen = {0: 1}` before the loop |
| Forgetting negative numbers invalidate sliding window | Prefix + HashMap handles negatives; sliding window doesn't |

# 0019 Running Sum of 1d Array

In [13]:
"""
id: lc_1480
title: Running Sum of 1d Array
source: leetcode
difficulty: easy
primary: prefix sum
tags: [array, prefix-sum]
leetcode_url: https://leetcode.com/problems/running-sum-of-1d-array/
status: draft
last_updated: 2026-04-23
notes:
- key idea: compute cumulative sum at each index
- time: O(n)
- space: O(1) excluding output array
"""

# ============================================================================
# File: 1480_lc_1480_running_sum_of_1d_array_empty.py
# LeetCode 1480: Running Sum of 1d Array
# Difficulty: Easy
#
# PROBLEM STATEMENT:
# Given an array nums, return a new array result where result[i] is the sum 
# of all elements from index 0 to i (inclusive).
#
# RULES:
# - nums.length is between 1 and 1000.
# - result[i] = sum(nums[0...i]).
#
# EXAMPLES:
# Input: nums = [1,2,3,4]
# Output: [1,3,6,10]
#
# Input: nums = [1,1,1,1,1]
# Output: [1,2,3,4,5]
# ============================================================================

from typing import List

def runningSum(nums: List[int]) -> List[int]:
    prevSum = 0
    out = [0] * len(nums)
    for i, num in enumerate(nums):
        prevSum+= num
        out[i] = prevSum
    return out
print(runningSum([1, 2, 3, 4]))
print(runningSum([3, 1, 4, 1, 5]))

def test():
    assert runningSum([1, 2, 3, 4]) == [1, 3, 6, 10]  # given example 1
    assert runningSum([1, 1, 1, 1, 1]) == [1, 2, 3, 4, 5]  # given example 2
    assert runningSum([3, 1, 4, 1, 5]) == [3, 4, 8, 9, 14]  # given example 3
    assert runningSum([0]) == [0]  # single zero
    assert runningSum([10]) == [10]  # single positive
    assert runningSum([-1, -2, -3]) == [-1, -3, -6]  # negative numbers
    assert runningSum([0, 0, 0]) == [0, 0, 0]  # multiple zeros
    assert runningSum([1, -1, 1, -1]) == [1, 0, 1, 0]  # alternating
    assert runningSum([100, 200]) == [100, 300]  # simple pair
    assert runningSum([-5, 10, -5]) == [-5, 5, 0]  # sum back to zero
    print('All Pass!')

test()

[1, 3, 6, 10]
[3, 4, 8, 9, 14]
All Pass!


# 0069 Find Pivot Index

In [14]:
"""
id: lc_0724
title: Find Pivot Index
source: leetcode
difficulty: easy
primary: prefix-sum
tags: [array, prefix-sum]
leetcode_url: https://leetcode.com/problems/find-pivot-index/
status: draft
last_updated: 2026-04-23
notes:
- key idea: total_sum - left_sum - current_val == left_sum
- time: O(n)
- space: O(1)
"""

# ============================================================================
# File: 724_lc_0724_find_pivot_index_empty.py
# LeetCode 724: Find Pivot Index
# Difficulty: Easy
#
# PROBLEM STATEMENT:
# Given an integer array nums, return the leftmost pivot index.
# The pivot index is an index where the sum of all numbers to the left
# equals the sum of all numbers to the right.
#
# RULES:
# - Numbers at the pivot itself are not included in either sum.
# - If index is 0, left sum is 0.
# - If index is n-1, right sum is 0.
# - Return -1 if no pivot index exists.
#
# EXAMPLES:
# nums = [1,7,3,6,5,6] -> 3 (11 == 11)
# nums = [1,2,3] -> -1
# nums = [2,1,-1] -> 0 (0 == 0)
# ============================================================================

from typing import List

def pivotIndex(nums: List[int]) -> int:
    prevSum = 0 
    total = sum(nums)

    for i, num in enumerate(nums):
        following_sum = total - prevSum - num
        if following_sum == prevSum:
            return i
        prevSum += num
    return -1
        
print(pivotIndex([1, 7, 3, 6, 5, 6]))   #3
print(pivotIndex([1, 2, 3])) # -1

def test():
    assert pivotIndex([1, 7, 3, 6, 5, 6]) == 3   # example 1: standard case
    assert pivotIndex([1, 2, 3]) == -1           # example 2: no pivot
    assert pivotIndex([2, 1, -1]) == 0           # example 3: pivot at start
    assert pivotIndex([-1, -1, -1, -1, -1, 0]) == 2 # negatives with 0
    assert pivotIndex([-1, -1, -1, 0, 1, 1]) == 0   # pivot at start with negatives
    assert pivotIndex([5]) == 0                  # single element
    assert pivotIndex([1, -1, 1]) == 0           # zero sum symmetry

    assert pivotIndex([1, 2, 1]) == 1            # simple symmetry
    assert pivotIndex([0, 0, 0, 0]) == 0         # all zeros leftmost requirement
    assert pivotIndex([-1, -1, 0, 1, 1]) == -1    # zero in middle
    print('All Pass!')

test()

3
-1
All Pass!


# 0023 Product Except self

In [15]:
"""
id: lc_0238
title: Product of Array Except Self
source: leetcode
difficulty: medium
primary: array
tags: [prefix-sum, suffix-sum, array]
leetcode_url: https://leetcode.com/problems/product-of-array-except-self/
status: draft
last_updated: 2026-04-23
notes:
- key idea: result[i] = prefix_product[i-1] * suffix_product[i+1]
- time: O(n)
- space: O(1) excluding output array
"""

# ============================================================================
# File: 238_lc_0238_product_except_self_empty.py
# LeetCode 238: Product of Array Except Self
# Difficulty: Medium
#
# PROBLEM STATEMENT:
# Given an array of integers nums, return an array result where result[i] is 
# the product of all elements except nums[i].
#
# RULES:
# - You must not use division.
# - Time complexity must be O(n) or O(n^2).
# - Input array is non-empty (2 <= nums.length <= 10^5).
#
# EXAMPLES:
# nums = [1,2,3,4] -> [24,12,8,6]
# nums = [-1,1,0,-3,3] -> [0,0,9,0,0]
# nums = [2,3] -> [3,2]
# ============================================================================

from typing import List

def productExceptSelf(nums: List[int]) -> List[int]:
    out = [1] * len(nums)
    n = len(nums)
    prefix, postfix = 1,1
    for i in range(n):
        out[i] = prefix
        prefix *= nums[i]
    for i in range(n-1, -1, -1):
        out[i] *= postfix
        postfix *= nums[i]
    return out

print(productExceptSelf([1, 1, 1]))
print(productExceptSelf([1, 2, 3, 4]))
print(productExceptSelf([-1, 1, 0, -3, 3]))

def test():
    assert productExceptSelf([1, 2, 3, 4]) == [24, 12, 8, 6]      # example 1: standard
    assert productExceptSelf([-1, 1, 0, -3, 3]) == [0, 0, 9, 0, 0] # example 2: single zero
    assert productExceptSelf([2, 3]) == [3, 2]                   # example 3: minimal length
    assert productExceptSelf([0, 0]) == [0, 0]                   # two zeros
    
    assert productExceptSelf([5, 2, 1]) == [2, 5, 10]            # decreasing values
    assert productExceptSelf([-1, -1, -1]) == [1, 1, 1]          # all negative ones
    assert productExceptSelf([1, 0, 0, 1]) == [0, 0, 0, 0]       # multiple zeros
    assert productExceptSelf([4, 3, 2, 1]) == [6, 8, 12, 24]     # reversed example 1
    assert productExceptSelf([1, 5]) == [5, 1]                   # small case with 1
    assert productExceptSelf([1, 1, 1]) == [1, 1, 1]              # all ones
    print('All Pass!')

test()

[1, 1, 1]
[24, 12, 8, 6]
[0, 0, 9, 0, 0]
All Pass!


# 0035   Subarray Sum Equals K

In [16]:
"""
id: lc_0560
title: Subarray Sum Equals K
source: leetcode
difficulty: medium
primary: hash-table
tags: [array, hash-table, prefix-sum]
leetcode_url: https://leetcode.com/problems/subarray-sum-equals-k/
status: draft
last_updated: 2026-04-23
notes:
- key idea: current_sum - k in counts_of_previous_prefix_sums
- time: O(n)
- space: O(n)
"""

# ============================================================================
# File: 560_lc_0560_subarray_sum_equals_k_empty.py
# LeetCode 560: Subarray Sum Equals K
# Difficulty: Medium
#
# PROBLEM STATEMENT:
# Given an array of integers nums and an integer k, return the number of 
# contiguous subarrays whose sum equals k.
#
# RULES:
# - Subarrays must be contiguous.
# - Array can include negative numbers.
# - Time complexity should be O(n^2) or better.
#
# EXAMPLES:
# nums = [1,1,1], k = 2 -> 2
# nums = [1,2,3], k = 3 -> 2
# nums = [1,-1,1], k = 1 -> 3
# ============================================================================

from typing import List
from collections import defaultdict
def subarraySum(nums: List[int], k: int) -> int:
    seen = defaultdict(int)
    seen[0] = 1
    runSum = 0
    ret = 0
    for num in nums:
        runSum += num
        ret += seen[runSum-k]
        seen[runSum] += 1
    return ret


print(subarraySum([1, 1, 1], 2))   #2
print(subarraySum([1, 2, 3], 3))

def test():
    assert subarraySum([1, 1, 1], 2) == 2        # example 1: overlapping
    assert subarraySum([1, 2, 3], 3) == 2        # example 2: different lengths
    
    assert subarraySum([1], 0) == 0              # single element no match
    assert subarraySum([1], 1) == 1              # single element match
    
    assert subarraySum([-1, -1, 1], 0) == 1      # negative and positive cancel
    
    assert subarraySum([10, 2, -2, -20, 10], -10) == 3 # large jumps
    assert subarraySum([3, 4, 7, 2, -3, 1, 4, 2], 7) == 4 # complex case
    assert subarraySum([1, -1, 1], 1) == 3       # example 3: negatives involved
    assert subarraySum([0, 0, 0, 0], 0) == 10    # all zeros
    assert subarraySum([1, 2, 3, -3, 3], 3) == 5 # multiple paths to sum
    print('All Pass!')

test()


2
2
All Pass!


# 0158 Range Sum Query - Immutable

In [17]:

# We keep one extra 0 at the front of prefix sums.
# That shifts everything by 1, so range sum [left..right]
# is always prefix[right+1] - prefix[left] (no special case for left=0).

# TODO: add problem statement, solution, and tests
"""
id: lc_0303
title: Range Sum Query - Immutable
source: leetcode
difficulty: easy
primary: prefix-sum
tags: [array, design, prefix-sum]
leetcode_url: https://leetcode.com/problems/range-sum-query-immutable/
status: draft
last_updated: 2026-04-23
notes:
- key idea: P[i] = sum(nums[0...i-1]), then sum(left...right) = P[right+1] - P[left]
- time: O(n) for constructor, O(1) per query
- space: O(n) to store prefix sums
"""

# ============================================================================
# File: 303_lc_0303_range_sum_query_immutable_empty.py
# LeetCode 303: Range Sum Query - Immutable
# Difficulty: Easy
#
# PROBLEM STATEMENT:
# Given an integer array nums, handle multiple queries of the following type:
# Calculate the sum of the elements of nums between indices left and right 
# inclusive where left <= right.
#
# RULES:
# - Implement the NumArray class.
# - sumRange(left, right) must return the sum of nums[left...right].
# - Optimization for multiple calls is expected (O(1) query time).
#
# EXAMPLES:
# nums = [-2, 0, 3, -5, 2, -1]
# sumRange(0, 2) -> 1
# sumRange(2, 5) -> -1
# sumRange(0, 5) -> -3
# ============================================================================

from typing import List

from typing import List

class NumArray:
    def __init__(self, nums: List[int]):
        self._data = [0] * (len(nums) + 1)
        current_sum = 0
        for i, n in enumerate(nums):
            current_sum += n
            self._data[i + 1] = current_sum

    def sumRange(self, left: int, right: int) -> int:
        return self._data[right + 1] - self._data[left]


# Dummy instances for demo print calls
obj = NumArray([-2, 0, 3, -5, 2, -1])
print(obj.sumRange(0, 2))  # 1
print(obj.sumRange(2, 5))   # -1


def test():
    a = NumArray([-2, 0, 3, -5, 2, -1])
    assert a.sumRange(0, 2) == 1
    assert a.sumRange(2, 5) == -1
    assert a.sumRange(0, 5) == -3

    b = NumArray([1, 2, 3, 4])
    assert b.sumRange(0, 0) == 1
    assert b.sumRange(3, 3) == 4
    assert b.sumRange(1, 2) == 5
    assert b.sumRange(0, 3) == 10

    c = NumArray([-1])
    assert c.sumRange(0, 0) == -1

    d = NumArray([10, -10, 10, -10])
    assert d.sumRange(0, 3) == 0
    assert d.sumRange(1, 2) == 0

    print('All Pass!')

test()

1
-1
All Pass!


# 0159

In [18]:
"""
id: lc_0325
title: Maximum Size Subarray Sum Equals k
source: leetcode
difficulty: medium
primary: hash map
tags: [array, hash-table, prefix-sum]
leetcode_url: https://leetcode.com/problems/maximum-size-subarray-sum-equals-k/
status: draft
last_updated: 2026-04-23
notes:
- key idea: prefix sums with hash map to store first occurrence of each sum
- time: O(n)
- space: O(n)
"""

# ============================================================================
# File: 325_lc_0325_maximum_size_subarray_sum_equals_k_empty.py
# LeetCode 325: Maximum Size Subarray Sum Equals k
# Difficulty: Medium
#
# PROBLEM STATEMENT:
# Given an array nums and a target value k, find the maximum length of a 
# subarray that sums to k. If there isn't one, return 0 instead.
#
# RULES:
# - The sum of the entire subarray must exactly equal k.
# - Return the maximum length (integer).
#
# EXAMPLES:
# Input: nums = [1, -1, 5, -2, 3], k = 3
# Output: 4 (Subarray [1, -1, 5, -2] sums to 3)
#
# Input: nums = [-2, -1, 2, 1], k = 1
# Output: 2 (Subarray [-1, 2] sums to 1)
# ============================================================================

from typing import List
from collections import defaultdict
def maxSubArrayLen(nums: List[int], k: int) -> int:
    # TODO: implement
    seen = defaultdict(int)
    prefix = 0
    seen[prefix] = -1
    best = 0
    for i, num in enumerate(nums):
        prefix += num
        delta = prefix - k
        if delta in seen:
            best = max(best , i - seen[delta])
            
        if prefix not in seen:    #keep only the first seen because we nee max length
            seen[prefix] = i
        
    return best

print(maxSubArrayLen([1, -1, 5, -2, 3], 3))  # 4
print(maxSubArrayLen([-2, -1, 2, 1], 1))

def test():
    assert maxSubArrayLen([1, -1, 5, -2, 3], 3) == 4  # given example 1
    assert maxSubArrayLen([-2, -1, 2, 1], 1) == 2    # given example 2
    assert maxSubArrayLen([1, 1, 0], 1) == 2         # include zero
    assert maxSubArrayLen([10, 5, 2, 7, 1, 9], 15) == 4 # middle subarray
    assert maxSubArrayLen([-1], -1) == 1             # single element match
    assert maxSubArrayLen([1, 2, 3], 10) == 0        # no match
    assert maxSubArrayLen([0, 0, 0], 0) == 3         # all zeros target zero
    assert maxSubArrayLen([1, -1, 1, -1], 0) == 4    # full array sum zero
    assert maxSubArrayLen([5], 5) == 1               # exact single match
    assert maxSubArrayLen([2, -2, 2, -2], 0) == 4    # oscillating sums
    print('All Pass!')

test()

4
2
All Pass!


# 0160   LC 525 - Contiguous Array

In [19]:
# treat 0 as -1
# find longest subarray with k = 0

"""
id: lc_0525
title: Contiguous Array
source: leetcode
difficulty: medium
primary: hash_table
tags: [prefix_sum, hash_table, array]
leetcode_url: https://leetcode.com/problems/contiguous-array/
status: draft
last_updated: 2026-04-23
notes:
- key idea: convert 0s to -1s and find longest subarray with sum 0 using prefix sum map
- time: O(n)
- space: O(n)
"""

# ============================================================================
# File: 525_lc_0525_contiguous_array_empty.py
# LeetCode 525: Contiguous Array
# Difficulty: Medium
#
# PROBLEM STATEMENT:
# Given a binary array nums, return the maximum length of a contiguous 
# subarray with an equal number of 0 and 1.
#
# RULES:
# - nums consists only of 0 and 1.
# - The length of the subarray must be even or zero.
#
# EXAMPLES:
# Input: nums = [0,1] -> Output: 2
# Input: nums = [0,1,0] -> Output: 2
# ============================================================================

from typing import List
from collections import defaultdict
def findMaxLength(nums: List[int]) -> int:
    k = 0 
    seen = defaultdict(int)
    prefix = 0
    seen[prefix] = -1
    best = 0
    for i, num in enumerate(nums):
        if num == 0:
            val = -1
        else:
            val = num
        
        prefix += val
        if prefix in seen:
            best = max(best , i - seen[prefix])
            
        if prefix not in seen:    #keep only the first seen because we nee max length
            seen[prefix] = i
        
    return best

print(findMaxLength([0, 1]))
print(findMaxLength([0, 1, 0]))

def test():
    assert findMaxLength([0, 1]) == 2  # given example 1 — basic pair
    assert findMaxLength([0, 1, 0]) == 2  # given example 2 — ignore trailing 0
    assert findMaxLength([0, 0, 1, 1, 0]) == 4  # mid-array match
    assert findMaxLength([0, 0, 0, 1, 1, 1]) == 6  # full array match
    assert findMaxLength([1, 1, 1, 1]) == 0  # no balance possible
    assert findMaxLength([0, 0, 0]) == 0  # all zeros
    assert findMaxLength([0, 1, 1, 0, 1, 1, 0, 0]) == 8  # interleaved balance
    assert findMaxLength([1]) == 0  # single element
    assert findMaxLength([]) == 0  # empty array
    assert findMaxLength([0, 1, 1, 1, 0, 0]) == 6  # complex balance
    print('All Pass!')

test()

2
2
All Pass!


## 0161  Not doing it now for difficulty